In [147]:
import pandas as pd
import numpy as np
from tools import sherlock

In [148]:
df = pd.read_excel(r"..\data\raw\consultations_janvier2026.xlsx", sheet_name=0, skiprows=1, nrows=20)

# Exploration

In [149]:
sherlock(df)

20 lignes | 4 colonnes | 0 lignes doublons
--------------------------------------------------
                      Type  Manquants Manquants %  Uniques
Mois                   str          0        0.0%       20
Consultations        int64          0        0.0%       19
Réservations         int64          0        0.0%       10
Taux de réservation    str          0        0.0%       14
--------------------------------------------------
Clé potentielle sur Mois


In [150]:
df = df.rename(str.lower, axis='columns')
df.columns = df.columns.str.strip().str.replace(' ', '_')
df['taux_de_réservation'] = df['taux_de_réservation'].str.strip().str.replace('%', '').astype(float)

In [151]:
df['period_label'] = df['mois']
mois_fr = {
    "Janvier": 1, "Février": 2, "Mars": 3, "Avril": 4,
    "Mai": 5, "Juin": 6, "Juillet": 7, "Août": 8,
    "Septembre": 9, "Octobre": 10, "Novembre": 11, "Décembre": 12
}
df['mois_num'] = df['mois'].str.split().str[0].map(mois_fr).astype(str)
df['année'] = df['mois'].str.split().str[1]
df['mois'] = df['mois_num'] + " " + df['année']
df['mois'] = pd.to_datetime(df['mois'], format="%m %Y").dt.to_period('M')
df['année'] = df['année'].astype(int)
df = df.drop(columns=['mois_num'])

In [152]:
sherlock(df)

20 lignes | 6 colonnes | 0 lignes doublons
--------------------------------------------------
                          Type  Manquants Manquants %  Uniques
mois                 period[M]          0        0.0%       20
consultations            int64          0        0.0%       19
réservations             int64          0        0.0%       10
taux_de_réservation    float64          0        0.0%       14
period_label               str          0        0.0%       20
année                    int64          0        0.0%        3
--------------------------------------------------
Clé potentielle sur mois, period_label


In [153]:
df

,mois,consultations,réservations,taux_de_réservation,period_label,année
0,2024-07,680,17,2.5,Juillet 2024,2024
1,2024-08,352,8,2.3,Août 2024,2024
2,2024-09,442,9,2.0,Septembre 2024,2024
3,2024-10,308,2,0.6,Octobre 2024,2024
4,2024-11,124,0,0.0,Novembre 2024,2024
5,2024-12,190,5,2.6,Décembre 2024,2024
6,2025-01,253,5,2.0,Janvier 2025,2025
7,2025-02,372,5,1.3,Février 2025,2025
8,2025-03,529,15,2.8,Mars 2025,2025
9,2025-04,245,4,1.6,Avril 2025,2025


# Feature

Consultations : variation en % entre le mois M et le mois M-1 + Y et Y-1

In [154]:
df['var_consultations_pct'] = (df['consultations'] - df['consultations'].shift(1)) / df['consultations'].shift(1) * 100
df['var_consultations_pct_annuel'] = (df['consultations'] - df['consultations'].shift(12)) / df['consultations'].shift(12) * 100

,mois,consultations,réservations,taux_de_réservation,period_label,année,var_consultations_pct,var_consultations_pct_annuel
0,2024-07,680,17,2.5,Juillet 2024,2024,NaN,NaN
1,2024-08,352,8,2.3,Août 2024,2024,-48.235294,NaN
2,2024-09,442,9,2.0,Septembre 2024,2024,25.568182,NaN
3,2024-10,308,2,0.6,Octobre 2024,2024,-30.316742,NaN
4,2024-11,124,0,0.0,Novembre 2024,2024,-59.740260,NaN
5,2024-12,190,5,2.6,Décembre 2024,2024,53.225806,NaN
6,2025-01,253,5,2.0,Janvier 2025,2025,33.157895,NaN
7,2025-02,372,5,1.3,Février 2025,2025,47.035573,NaN
8,2025-03,529,15,2.8,Mars 2025,2025,42.204301,NaN
9,2025-04,245,4,1.6,Avril 2025,2025,-53.686200,NaN


Réservation & tx de résa : variation en % et point entre Y et Y-1

In [158]:
df['var_resa_pct_annuel'] = (df['réservations'] - df['réservations'].shift(12)) / df['réservations'].shift(12) * 100
df['var_resa_pts_annuel'] = (df['taux_de_réservation'] - df['taux_de_réservation'].shift(12))
df = df.replace(np.inf, np.nan)
df

,mois,consultations,réservations,taux_de_réservation,period_label,année,var_consultations_pct,var_consultations_pct_annuel,var_resa_pct_annuel,var_resa_pts_annuel
0,2024-07,680,17,2.5,Juillet 2024,2024,NaN,NaN,NaN,NaN
1,2024-08,352,8,2.3,Août 2024,2024,-48.235294,NaN,NaN,NaN
2,2024-09,442,9,2.0,Septembre 2024,2024,25.568182,NaN,NaN,NaN
3,2024-10,308,2,0.6,Octobre 2024,2024,-30.316742,NaN,NaN,NaN
4,2024-11,124,0,0.0,Novembre 2024,2024,-59.740260,NaN,NaN,NaN
5,2024-12,190,5,2.6,Décembre 2024,2024,53.225806,NaN,NaN,NaN
6,2025-01,253,5,2.0,Janvier 2025,2025,33.157895,NaN,NaN,NaN
7,2025-02,372,5,1.3,Février 2025,2025,47.035573,NaN,NaN,NaN
8,2025-03,529,15,2.8,Mars 2025,2025,42.204301,NaN,NaN,NaN
9,2025-04,245,4,1.6,Avril 2025,2025,-53.686200,NaN,NaN,NaN


### Export

In [159]:
df.to_csv(r"..\data\processed\airbnb.csv", index=False, sep=';', decimal=',', encoding='utf-8-sig')